In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import math

In [ ]:
smesh = pd.read_csv("concatSmeshData/f864_airQualityMetrics.csv")
smesh.sort_values(by = "datetime", inplace = True, ignore_index = True)
smesh['datetime'] = pd.to_datetime(smesh['datetime'])
display(smesh.head(2))
smeshStart = smesh.iloc[0]["datetime"]
smeshEnd = smesh.iloc[-1]["datetime"]

print(smeshStart)
print(smeshEnd)

In [ ]:
purpleAir = pd.read_csv("purpleAirData/2025-10-23_17-30_us-epa-pm25-aqi-many.csv")
purpleAir.sort_values(by = "DateTime", inplace = True, ignore_index = True)
purpleAir['datetime'] = pd.to_datetime(purpleAir['DateTime'])

purpleAir = purpleAir[["datetime","Pepperwood Preserve A","Pepperwood Preserve B","MWSRPINGS A","MWSRPINGS B","8566 outside A","8566 outside B","8566 inside A"]]
display(purpleAir.head(2))
purpleAirStart = purpleAir.iloc[0]["datetime"]
purpleAirEnd = purpleAir.iloc[-1]["datetime"]


print(purpleAirStart)
print(purpleAirEnd)

In [ ]:
start = max(purpleAirStart, smeshStart)
end = min(purpleAirEnd, smeshEnd)
print(start, end)

In [ ]:
cutSmesh = smesh[(smesh["datetime"] >= start) & (smesh["datetime"] <= end+datetime.timedelta(minutes=10))].copy()
cutSmesh["10 Minute"] = pd.to_datetime(cutSmesh["datetime"]).dt.floor("10min")   # "10T" means 10 minutes
# smeshAveraged = cutSmesh.groupby("10 Minute", as_index=False)["pm25Standard"].mean()
smeshAveraged = cutSmesh.groupby("10 Minute", as_index=False)["pm25Environmental"].mean()

smeshAveraged["datetime"] = smeshAveraged["10 Minute"]
smeshAveraged = smeshAveraged.fillna(0)

display(smeshAveraged.head())


In [ ]:
fig,ax = plt.subplots(2,1)
plt.subplots_adjust(top = 0.99, bottom=0.01, hspace=0.27, wspace=0.4)

ax[0].plot(purpleAir["datetime"], purpleAir["MWSRPINGS A"])
ax[0].set_title("Purple Air PM 2.5")
ax[0].set_ylabel("PM 2.5")

ax[1].plot(smeshAveraged["datetime"], smeshAveraged["pm25Environmental"])
ax[1].set_title("Smesh PM 2.5")
ax[1].set_xlabel("Time")
ax[1].set_ylabel("PM 2.5")


# ax[1].plot(purpleAir["datetime"], purpleAir["Pepperwood Preserve B"])
# ax[1].set_title("Purple Air Channel B")
# ax[1].set_ylabel("PM25")
# ax[1].set_yscale("log")

# ax[1].plot(purpleAir["datetime"], (purpleAir["Pepperwood Preserve B"]))
# ax[1].set_title("Purple Air Channel B PM 2.5")
# ax[1].set_ylabel("PM25")



# ax[2].plot(smeshAveraged["datetime"], smeshAveraged["pm25Environmental"])
# ax[2].set_title("Smesh PM 2.5")
# ax[2].set_xlabel("Time")
# ax[2].set_ylabel("PM25")



plt.show()

In [ ]:
columns = ["Pepperwood Preserve A","Pepperwood Preserve B","MWSRPINGS A","MWSRPINGS B","8566 outside A","8566 outside B","8566 inside A"]

In [ ]:
fig,ax = plt.subplots(len(columns) + 1,1)

plt.subplots_adjust(top = 3, bottom=0.01, hspace=0.4, wspace=0.4)
for i in range(len(columns)):
    ax[i].plot(purpleAir["datetime"], purpleAir[columns[i]])
    ax[i].set_title(columns[i])
    ax[i].set_ylabel("PM 2.5")

ax[len(columns)].plot(smeshAveraged["datetime"], smeshAveraged["pm25Environmental"])
ax[len(columns)].set_title("Smesh PM 2.5")
ax[len(columns)].set_xlabel("Time")
ax[len(columns)].set_ylabel("PM 2.5")


In [ ]:
from statsmodels.tsa.stattools import adfuller
#Checks if stationary, if the mean and variance stay roughly consistent throughout the timeseries
#Which doesn't seem true
print(adfuller(purpleAir["MWSRPINGS A"])[1]) #P-value
print(adfuller(smeshAveraged["pm25Environmental"])[1]) #P-value


In [ ]:
print(len(smeshAveraged))
print(len(purpleAir))


In [ ]:
difference = []
logDifference = []
for i in range(len(purpleAir["MWSRPINGS A"])):
    purplePM25 = purpleAir["MWSRPINGS A"][i]
    smeshPM25 = smeshAveraged["pm25Environmental"][i]

    difference.append(purplePM25-smeshPM25)
    if np.isnan(smeshPM25) or smeshPM25 == 0:
        smeshPM25 = 1
    if np.isnan(purplePM25) or purplePM25 == 0:
        purplePM25 = 1
    logDifference.append(np.log(purplePM25)-np.log(smeshPM25))

In [ ]:
plt.plot(difference)

In [ ]:
plt.plot(logDifference)

In [ ]:
print(adfuller(difference)[1]) #P-value
cleanLogDifference = [x for x in logDifference if not np.isnan(x)]

print(adfuller(cleanLogDifference)[1]) #P-value